# AudioAssistent: ручное тестирование
**Часть A. Локально** (без сервера и Postgres): аудио → транскрипт → говорящие → статус. Нужна только Ollama (и mlx-whisper для аудио).
**Часть B. Ручки сервиса** (нужны запущенные `uvicorn app.main:app` и PostgreSQL): каждая стадия отдельно, просмотр запуска, удаление.

Запись кладём в `data/audio/` под именем `YYYYMMDDHHMMSS.WAV` (из имени берутся дата и время встречи).

In [ ]:
import sys, json
from pathlib import Path
from datetime import date

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from observability import init_observability
init_observability()                      # структурные логи; в ноутбуке удобно LOG_FORMAT=console

from audio_assistant.cli import build_local_pipeline   # Ollama + файловая история, без Postgres
from audio_assistant.client.pipeline import load_transcript
from audio_assistant.client.domain import Transcript
from audio_assistant.client.recordings import list_recordings, parse_recording_datetime

pipeline, settings = build_local_pipeline()
print("Аудитории:", pipeline.prompts.available())
print("Подсказка для Whisper:", pipeline.team.names_hint())

# Часть A. Локально

## A1. Записи и дата встречи

In [ ]:
recordings = list_recordings(settings.audio_dir)
for r in recordings:
    print(r["file_name"], r["size_bytes"], "байт | время записи:", r["recorded_at"])
AUDIO = settings.audio_dir / recordings[0]["file_name"] if recordings else None
DAY = (parse_recording_datetime(AUDIO.name).date() if AUDIO and parse_recording_datetime(AUDIO.name) else date.today())
print("Берём:", AUDIO, "| дата встречи:", DAY)

## A2. Аудио → сырой транскрипт (Whisper). Результат кэшируется, чтобы не транскрибировать заново

In [ ]:
FORCE = False   # True — перетранскрибировать
cache = settings.output_dir / f"{AUDIO.stem}.raw.json" if AUDIO else None
if AUDIO and cache.exists() and not FORCE:
    raw = Transcript.from_dict(json.loads(cache.read_text(encoding="utf-8")))
    print("из кэша:", cache.name)
elif AUDIO:
    settings.output_dir.mkdir(parents=True, exist_ok=True)
    raw = pipeline.transcribe(AUDIO)
    cache.write_text(json.dumps(raw.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
else:
    raw = load_transcript(ROOT / "examples" / "sample_transcript.txt")   # запасной вариант без аудио
    print("аудио нет, берём examples/sample_transcript.txt")
for s in raw.segments:
    print(f"{s.start:6.1f}-{s.end:6.1f}  {s.text}")

## A3. Разметка говорящих
Два метода: **llm** (модель размечает чанками; имена распознаёт код, модель решает, передаёт ли упоминание слово) и **rules** (регулярные выражения). `— ведущий —` — речь вне меток.

In [ ]:
METHOD = "llm"          # llm | rules
attributed = pipeline.attribute(raw, METHOD)
print("метод:", attributed.meta.get("method"), "| чанков:", attributed.meta.get("chunks"),
      "| чанки, размеченные правилами из-за сбоя модели:", attributed.meta.get("fallback_chunks"))
print(attributed.render())

### Сравнение методов: чья реплика чья

In [ ]:
by_llm = pipeline.attribute(raw, "llm").text_by_speaker()
by_rules = pipeline.attribute(raw, "rules").text_by_speaker()
print(f"{'говорящий':28} {'llm':>5} {'rules':>6}   (слов)")
for who in dict.fromkeys([*by_llm, *by_rules]):
    print(f"{(who or '— ведущий —'):28} {len(by_llm.get(who, '').split()):>5} {len(by_rules.get(who, '').split()):>6}")

In [ ]:
spoken = attributed.text_by_speaker()
print("Без реплик:", [p.full_name for p in pipeline.team.participants if p.full_name not in spoken] or "нет")

## A4. Статус для аудитории
`save=False` — не писать историю. Правьте `prompts/<аудитория>/` и перезапускайте ячейку.

In [ ]:
AUDIENCE = "management"      # management | project_managers | team
res = pipeline.run_transcript(attributed, AUDIENCE, DAY, save=False)
print(res.text)

In [ ]:
# Сравнение аудиторий (по запросу к LLM на каждого говорившего)
for aud in pipeline.prompts.available():
    print("=" * 30, aud, "=" * 30)
    print(pipeline.run_transcript(attributed, aud, DAY, save=False).text)

## A5. Один сотрудник, промпт целиком (отладка)

In [ ]:
from string import Template
profile = pipeline.prompts.get(AUDIENCE)
who = next(p for p in pipeline.team.participants if p.full_name in spoken)
user = Template(profile.person_template).safe_substitute(
    date=DAY.strftime("%d.%m.%Y"), project_id=who.project_id, full_name=who.full_name,
    previous_status="нет данных", examples=profile.examples, utterance=spoken[who.full_name])
print(profile.system, "\n--------\n", user)

# Часть B. Ручки сервиса
Запуск: `POSTGRES_PORT=… uvicorn app.main:app --port 8000`. Каждая стадия сохраняет результат в Postgres под `run_id`,
повторный вызов заменяет его, поэтому можно править алиасы/промпты и перепроверять только нужный шаг.

In [ ]:
import requests
BASE = "http://localhost:8000"

def api(method, path, **kw):
    r = requests.request(method, BASE + path, timeout=1800, **kw)
    try:
        body = r.json()
    except ValueError:
        body = r.text
    print(method, path, "->", r.status_code)
    return body

def show(obj, limit=3000):
    print(json.dumps(obj, ensure_ascii=False, indent=2)[:limit])

show(api("GET", "/assistant/health"))
show(api("GET", "/ollama/health"))
show(api("GET", "/postgres/health"))

## B1. Записи в `data/audio`

In [ ]:
files = api("GET", "/assistant/audio")
show(files)
FILE = files[0]["file_name"]

## B2. Стадия 1: транскрибация (создаёт `run_id`)

In [ ]:
t = api("POST", "/assistant/transcribe", json={"file_name": FILE})
RUN = t["run_id"]
print("run_id =", RUN, "| дата встречи:", t["meeting_date"], "| записано:", t["recorded_at"])
for s in t["segments"]:
    print(f"{s['start']:6.1f}-{s['end']:6.1f}  {s['text']}")

## B3. Стадия 2: разбивка по говорящим
Метод можно переключать и вызывать стадию повторно, чтобы сравнить: транскрибация не повторяется.

In [ ]:
METHOD = "llm"          # llm | rules | None (из настроек сервиса)
a = api("POST", f"/assistant/runs/{RUN}/attribute", json={"method": METHOD})
print("метод:", a["method"], "| чанков:", a["chunks"], "| размечено правилами из-за сбоя модели:", a["fallback_chunks"])
for u in a["dialogue"]:
    print(f"[{u['speaker'] or '— ведущий —'}] {u['text']}")
print("Без реплик:", a["without_utterances"])

## B4. Стадия 3: статусы по людям (LLM)

In [ ]:
AUD = "management"
s = api("POST", f"/assistant/runs/{RUN}/summarize", json={"audience": AUD})
for st in s["statuses"]:
    if st["has_data"]:
        print("-", st["full_name"], ":", st["text"])

## B5. Стадия 4: итоговый отчёт (без LLM)

In [ ]:
r = api("POST", f"/assistant/runs/{RUN}/report", json={"audience": AUD})
print(r["text"])

## B6. Всё по запуску и список запусков

In [ ]:
d = api("GET", f"/assistant/runs/{RUN}")
print("стадия:", d["run"]["status"], "| сырых сегментов:", len(d["raw_transcript"]),
      "| реплик:", len(d["dialogue"]), "| аудитории:", list(d["statuses"]), "| отчёты:", list(d["reports"]))
show(api("GET", "/assistant/runs?limit=5"))

## B7. Сквозные запуски
`/assistant/runs/{id}/process` — стадии 2–4 по уже сохранённому транскрипту; `/assistant/process-file` — всё одним вызовом.

In [ ]:
# r = api("POST", f"/assistant/runs/{RUN}/process", json={"audience": "team"})
# r = api("POST", "/assistant/process-file", json={"file_name": FILE, "audience": AUD})

## B8. Удаление
Сначала всегда `dry_run=true`: покажет, сколько строк в каких таблицах будет удалено. Удаляются запуск и все связанные строки
(транскрипты, реплики, статусы, отчёты, журнал LLM); статусы удалённой встречи не попадают в «прошлый статус».

In [ ]:
show(api("DELETE", f"/assistant/runs/{RUN}", params={"dry_run": True}))                 # по id
show(api("DELETE", "/assistant/runs", params={"file_name": FILE, "dry_run": True}))      # по имени файла
show(api("DELETE", "/assistant/runs", params={"meeting_date": DAY.isoformat(), "dry_run": True}))  # по дате
show(api("DELETE", "/assistant/runs", params={"all": True, "dry_run": True}))            # всё

In [ ]:
CONFIRM_DELETE = False      # поставьте True, чтобы удалить по-настоящему
if CONFIRM_DELETE:
    show(api("DELETE", f"/assistant/runs/{RUN}"))
    # show(api("DELETE", "/assistant/runs", params={"file_name": FILE}))
    # show(api("DELETE", "/assistant/runs", params={"meeting_date": DAY.isoformat()}))
    # show(api("DELETE", "/assistant/runs", params={"all": True}))     # очищает ВСЕ таблицы